## Basics

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
!wget https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt

--2026-05-13 18:00:50--  https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36739 (36K) [text/plain]
Saving to: ‘tips_ML_Phenotype.txt’

tips_ML_Phenotype.t 100%[===================>]  35.88K  --.-KB/s    in 0.003s  

2026-05-13 18:00:50 (12.4 MB/s) - ‘tips_ML_Phenotype.txt’ saved [36739/36739]



In [4]:
with open('tips_ML_Phenotype.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  36358


In [6]:
print(text[:1000])

Phenotyping Data and ML
The enormous volume, variety, velocity, and veracity of imaging and remote-sensing data
generated by such real-time platforms represent a ‘big data’ problem. The data generated
by these near real-time platforms must be efﬁciently archived and retrieved for analysis. Although
the analysis and interpretation of such (image-based) big data are challenging, the ensuing
possibilities that can impact on agricultural production make it a promising approach for HTP
and HTSP. ML approaches present a scalable, modular strategy for data analysis, especially for
the new application domain of ‘plant stress analytics’. Recent studies on HTSP using images
obtained from UAV-based platforms to detect weeds in wheat (Triticum aestivum L.) [23], maize
[21], and sunﬂower (Helianthus annuus L.) [24] using ML algorithms have paved a new path for
better stress management practices on spatial and temporal basis.
ML is an inherently multidisciplinary approach to data analysis that draws

In [7]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 %'(),-./0123456789:;?ABCDEFGHIKLMNOPQRSTUVWY[]abcdefghijklmnopqrstuvwxyzï–‘’ﬁﬂ
81


In [8]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("plants are cool"))
print(decode(encode("plants are cool")))

[64, 60, 49, 62, 68, 67, 2, 49, 66, 53, 2, 51, 63, 63, 60]
plants are cool


Other tokenizers

https://huggingface.co/docs/transformers/main_classes/tokenizer

https://github.com/google/sentencepiece

In [9]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([36358]) torch.int64
tensor([38, 56, 53, 62, 63, 68, 73, 64, 57, 62, 55,  2, 27, 49, 68, 49,  2, 49,
        62, 52,  2, 35, 34,  0, 42, 56, 53,  2, 53, 62, 63, 66, 61, 63, 69, 67,
         2, 70, 63, 60, 69, 61, 53,  7,  2, 70, 49, 66, 57, 53, 68, 73,  7,  2,
        70, 53, 60, 63, 51, 57, 68, 73,  7,  2, 49, 62, 52,  2, 70, 53, 66, 49,
        51, 57, 68, 73,  2, 63, 54,  2, 57, 61, 49, 55, 57, 62, 55,  2, 49, 62,
        52,  2, 66, 53, 61, 63, 68, 53,  8, 67, 53, 62, 67, 57, 62, 55,  2, 52,
        49, 68, 49,  0, 55, 53, 62, 53, 66, 49, 68, 53, 52,  2, 50, 73,  2, 67,
        69, 51, 56,  2, 66, 53, 49, 60,  8, 68, 57, 61, 53,  2, 64, 60, 49, 68,
        54, 63, 66, 61, 67,  2, 66, 53, 64, 66, 53, 67, 53, 62, 68,  2, 49,  2,
        77, 50, 57, 55,  2, 52, 49, 68, 49, 78,  2, 64, 66, 63, 50, 60, 53, 61,
         9,  2, 42, 56, 53,  2, 52, 49, 68, 49,  2, 55, 53, 62, 53, 66, 49, 68,
        53, 52,  0, 50, 73,  2, 68, 56, 53, 67, 53,  2, 62, 53, 49, 66,  2, 66,
        

In [10]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [11]:
block_size = 8
train_data[:block_size+1]

tensor([38, 56, 53, 62, 63, 68, 73, 64, 57])

In [12]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([38]) the target: 56
when input is tensor([38, 56]) the target: 53
when input is tensor([38, 56, 53]) the target: 62
when input is tensor([38, 56, 53, 62]) the target: 63
when input is tensor([38, 56, 53, 62, 63]) the target: 68
when input is tensor([38, 56, 53, 62, 63, 68]) the target: 73
when input is tensor([38, 56, 53, 62, 63, 68, 73]) the target: 64
when input is tensor([38, 56, 53, 62, 63, 68, 73, 64]) the target: 57


In [13]:
torch.manual_seed(1337)

batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 2, 68, 66, 53, 61, 53, 62, 52],
        [49, 60, 73, 67, 57, 67,  2, 68],
        [ 2, 69, 67, 69, 49, 60, 60, 73],
        [ 0, 68, 53, 51, 56, 62, 57, 65]])
targets:
torch.Size([4, 8])
tensor([[68, 66, 53, 61, 53, 62, 52, 63],
        [60, 73, 67, 57, 67,  2, 68, 56],
        [69, 67, 69, 49, 60, 60, 73,  2],
        [68, 53, 51, 56, 62, 57, 65, 69]])
----
when input is [2] the target: 68
when input is [2, 68] the target: 66
when input is [2, 68, 66] the target: 53
when input is [2, 68, 66, 53] the target: 61
when input is [2, 68, 66, 53, 61] the target: 53
when input is [2, 68, 66, 53, 61, 53] the target: 62
when input is [2, 68, 66, 53, 61, 53, 62] the target: 52
when input is [2, 68, 66, 53, 61, 53, 62, 52] the target: 63
when input is [49] the target: 60
when input is [49, 60] the target: 73
when input is [49, 60, 73] the target: 67
when input is [49, 60, 73, 67] the target: 57
when input is [49, 60, 73, 67, 57] the target: 67
when input is [4

In [14]:
print(xb) # our input to the transformer

tensor([[ 2, 68, 66, 53, 61, 53, 62, 52],
        [49, 60, 73, 67, 57, 67,  2, 68],
        [ 2, 69, 67, 69, 49, 60, 60, 73],
        [ 0, 68, 53, 51, 56, 62, 57, 65]])


# Full code

In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

!wget https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt
with open('tips_ML_Phenotype.txt', 'r', encoding='utf-8') as f:
    text = f.read()

--2026-05-13 18:00:54--  https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36739 (36K) [text/plain]
Saving to: ‘tips_ML_Phenotype.txt.1’

tips_ML_Phenotype.t 100%[===================>]  35.88K  --.-KB/s    in 0.003s  

2026-05-13 18:00:54 (13.6 MB/s) - ‘tips_ML_Phenotype.txt.1’ saved [36739/36739]



In [16]:


# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

#  simple  model
class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = LanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

import time

start_time = time.time()

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total time taken: {elapsed_time} seconds")

0.211793 M parameters
step 0: train loss 4.5602, val loss 4.5523
step 100: train loss 2.7286, val loss 2.7150
step 200: train loss 2.5427, val loss 2.5376
step 300: train loss 2.4339, val loss 2.4410
step 400: train loss 2.3291, val loss 2.3468
step 500: train loss 2.1934, val loss 2.2396
step 600: train loss 2.0783, val loss 2.1593
step 700: train loss 1.9686, val loss 2.0673
step 800: train loss 1.8715, val loss 2.0059
step 900: train loss 1.8137, val loss 1.9707
step 1000: train loss 1.7213, val loss 1.9165
step 1100: train loss 1.6674, val loss 1.8565
step 1200: train loss 1.6058, val loss 1.8184
step 1300: train loss 1.5460, val loss 1.8148
step 1400: train loss 1.4982, val loss 1.7708
step 1500: train loss 1.4645, val loss 1.7696
step 1600: train loss 1.4070, val loss 1.7567
step 1700: train loss 1.3744, val loss 1.7340
step 1800: train loss 1.3520, val loss 1.7544
step 1900: train loss 1.3123, val loss 1.7528
step 2000: train loss 1.2834, val loss 1.7454
step 2100: train loss 1.

In [17]:
# import torch.nn.functional as F

# class MultiHeadAttention(nn.Module):
#     """ Fused Flash Attention implementation """

#     def __init__(self, n_head, n_embd, dropout, block_size):
#         super().__init__()
#         assert n_embd % n_head == 0
#         # Key, Query, Value projections for all heads at once
#         self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
#         # Output projection
#         self.c_proj = nn.Linear(n_embd, n_embd)
#         # Regularization
#         self.dropout = dropout
#         self.n_head = n_head
#         self.n_embd = n_embd
#         # Flash Attention handles the causal mask internally if is_causal=True

#     def forward(self, x):
#         B, T, C = x.shape # batch size, sequence length, embedding dimensionality (n_embd)

#         # Calculate query, key, values for all heads in batch
#         # q, k, v shape: (B, T, n_head, head_size)
#         q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

#         hs = C // self.n_head
#         q = q.view(B, T, self.n_head, hs).transpose(1, 2) # (B, nh, T, hs)
#         k = k.view(B, T, self.n_head, hs).transpose(1, 2) # (B, nh, T, hs)
#         v = v.view(B, T, self.n_head, hs).transpose(1, 2) # (B, nh, T, hs)

#         # ---- FLASH ATTENTION CORE ----
#         # This replaces the entire manual Head logic
#         out = F.scaled_dot_product_attention(
#             q, k, v,
#             attn_mask=None,
#             dropout_p=self.dropout if self.training else 0,
#             is_causal=True # This handles the 'tril' masking automatically!
#         )
#         # ------------------------------

#         out = out.transpose(1, 2).contiguous().view(B, T, C) # re-assemble heads
#         out = self.c_proj(out)
#         return out

In [18]:
# generate text
context = torch.zeros((1, 1), dtype=torch.long, device=device)



In [19]:
print(context)

tensor([[0]], device='cuda:0')


In [20]:
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


subders to discoverst in soybean Theypally, a Similarly,
many supervisiAGlity
in assupervised. Here, relolow roused reduction and richleVM method a nominative
models calibution-studicg elorated. In barley able an extimation, and given explore the Bayes craps werege used win as efﬁciaploying (conumbatied into applycinarly, classically
resulted with resis using hyperspectral imaging camera weeds. In another resultsulture the presence ors to an this ML to make displayed
diseases, has regioled is to discoverarching is the expressioney relarchical ruritures.
It is mage dataset approaches supervisimplitications of inte-sennead mixture models, plant supervised models and learn a murrimers is known an part praptio atticural conypach in ML tools).gen Practititicus [36] withorthout classical identiﬁcation and approaches) can also be classiﬁer) generally disciplines for the nominal mey [45].
Therefore reliable, relial
in preprocessing of plant spoth model robleverary formur

predictionea mize 

Part a

In [21]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
assert enc.decode(enc.encode("hello world")) == "hello world"

# To get the tokeniser corresponding to a specific model in the OpenAI API:
enc = tiktoken.encoding_for_model("gpt-4o")
print(enc)

<Encoding 'o200k_base'>


In [22]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

!wget https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt
with open('tips_ML_Phenotype.txt', 'r', encoding='utf-8') as f:
    text = f.read()

--2026-05-13 18:05:41--  https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36739 (36K) [text/plain]
Saving to: ‘tips_ML_Phenotype.txt.2’

tips_ML_Phenotype.t 100%[===================>]  35.88K  --.-KB/s    in 0.001s  

2026-05-13 18:05:41 (26.5 MB/s) - ‘tips_ML_Phenotype.txt.2’ saved [36739/36739]



In [23]:
import torch
encoding = tiktoken.get_encoding("cl100k_base")
vocab_size = encoding.n_vocab
print(vocab_size)
# encodings = encoding.encode(text)
# num_tokens = len(encodings)
# print(num_tokens)

# Train and test splits
data = torch.tensor(encoding.encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

100277


In [24]:


# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [25]:
#test get batch
xb, yb = get_batch('train')
# print(xb.shape)
# print(xb)
# print(yb.shape)
# print(yb)

In [26]:


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

#  simple  model
class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [27]:
# Cleanup memory to avoid Colab Out-of-Memory (OOM) errors
del model, m
torch.cuda.empty_cache()
# Init model
model = LanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

import time

start_time = time.time()

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total time taken: {elapsed_time} seconds")

13.137077 M parameters
step 0: train loss 11.6685, val loss 11.6547
step 100: train loss 6.3177, val loss 7.6651
step 200: train loss 5.5006, val loss 7.3235
step 300: train loss 4.4818, val loss 7.1178
step 400: train loss 3.5218, val loss 7.2031
step 500: train loss 2.7775, val loss 7.4244
step 600: train loss 2.1711, val loss 7.6083
step 700: train loss 1.7001, val loss 7.9203
step 800: train loss 1.2726, val loss 8.2907
step 900: train loss 0.9399, val loss 8.6290
step 1000: train loss 0.7228, val loss 8.7809
step 1100: train loss 0.5504, val loss 9.1789
step 1200: train loss 0.4382, val loss 9.3072
step 1300: train loss 0.3643, val loss 9.5489
step 1400: train loss 0.3235, val loss 9.7036
step 1500: train loss 0.2817, val loss 9.7934
step 1600: train loss 0.2629, val loss 9.9296
step 1700: train loss 0.2422, val loss 9.9740
step 1800: train loss 0.2353, val loss 10.1618
step 1900: train loss 0.2261, val loss 10.3182
step 2000: train loss 0.2122, val loss 10.3353
step 2100: train l

In [28]:
# generate text
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(context)
print("Text generation test")
print("-" * 20)
print(encoding.decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

tensor([[0]], device='cuda:0')
Text generation test
--------------------
! exposed to varying level of ozone.
Various pixel-classifying methods were compared such as LDA, K-means clustering, FPM-T2 (ﬁt
to a pattern multivariate image analysis combined with T2 statistics), and FPM-RSS (FPM
combined with residual sum of squares statistics) [53].

Box 1. Key Take-Away Points for Practitioners
Identiﬁcation
(i) A large variety of ML methods have been successfully applied to the disease identiﬁcation problem. This is an area
where preprocessing of images will be very useful.
(ii) There are two ways to frame a disease identiﬁcation problem for ML based on the amount of based on the best approach is to learn
 the nominal (healthy state) image background to conventional multiple examples for training the use of such images, Clustering
iﬁcant nominal (healthy) and diseased datasets are challenging, the ensuing
possibilities that can impact on agricultural production make it a promising approac

part b

In [29]:
def RoPE(x, head_size, device):
    # x shape: (B, T, head_size)
    B, T, d = x.shape

    # Calculate frequencies
    # theta_i = 10000^(-2(i-1)/d)
    inv_freq = 1.0 / (10000 ** (torch.arange(0, d, 2).float().to(device) / d))
    t = torch.arange(T, device=device).type_as(inv_freq)
    freqs = torch.einsum('i,j->ij', t, inv_freq) # (T, d/2)

    # Create sin/cos embeddings
    emb = torch.cat((freqs, freqs), dim=-1) # (T, d)
    cos = emb.cos() # (T, d)
    sin = emb.sin() # (T, d)

    # Rotate: [-x1, x0, -x3, x2...]
    x_rotated = torch.cat((-x[..., 1::2], x[..., 0::2]), dim=-1)

    # RoPE formula: x * cos + x_rotated * sin
    return (x * cos) + (x_rotated * sin)


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # apply rope
        k = RoPE(k, self.head_size, x.device)
        q = RoPE(q, self.head_size, x.device)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

#  simple  model
class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        # pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb #+ pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [30]:
# Cleanup memory to avoid Colab Out-of-Memory (OOM) errors
del model, m
torch.cuda.empty_cache()
# Init model
model = LanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

import time

start_time = time.time()

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total time taken: {elapsed_time} seconds")

13.135029 M parameters
step 0: train loss 11.6744, val loss 11.6936
step 100: train loss 6.0128, val loss 7.4410
step 200: train loss 4.7073, val loss 7.0846
step 300: train loss 3.4645, val loss 6.9696
step 400: train loss 2.5137, val loss 7.2153
step 500: train loss 1.7712, val loss 7.5877
step 600: train loss 1.2306, val loss 7.8885
step 700: train loss 0.8515, val loss 8.2692
step 800: train loss 0.5894, val loss 8.4907
step 900: train loss 0.4366, val loss 8.7074
step 1000: train loss 0.3572, val loss 8.9643
step 1100: train loss 0.3108, val loss 9.1777
step 1200: train loss 0.2797, val loss 9.2673
step 1300: train loss 0.2508, val loss 9.3286
step 1400: train loss 0.2408, val loss 9.6833
step 1500: train loss 0.2139, val loss 9.6637
step 1600: train loss 0.2049, val loss 9.8043
step 1700: train loss 0.2071, val loss 9.8670
step 1800: train loss 0.1949, val loss 9.8367
step 1900: train loss 0.1933, val loss 10.0393
step 2000: train loss 0.1878, val loss 10.1093
step 2100: train lo

In [31]:
# generate text
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(context)
print("Text generation test")
print("-" * 20)
print(encoding.decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

tensor([[0]], device='cuda:0')
Text generation test
--------------------
! as the inputs any label, etc.) from such variations based on
images obtained using hyperspectral camera. The DAR algorithm was efﬁciently model the decreasing
(Capsicum annuum L.) plants. Examplese.g., linear vs nonlinear) and training algorithm.

 several of such as UAV-based platform was equipped with a radial basis function ozone.
Abbreviations:].stress symptoms (i) identiﬁcation of four
data was used to optimize herbicide application such as. Preprocessing steps such as HTSP
is generally a new application expert was used to successfully in a maize and
a maize plant. However, generative models can be very successful for classiﬁcation tasks, such as distinguishing images into
particularly data are challenging, yield.
ML methods have been applied to learn patterns from the data spectacular results to similar patterns and, ultimatelyiﬁcation of quantiﬁcation, and Finally, yield.
ML methods work better for smalle

Part c

In [32]:
def RoPE(x, head_size, device):
    # x shape: (B, T, head_size)
    B, T, d = x.shape

    # Calculate frequencies
    # theta_i = 10000^(-2(i-1)/d)
    inv_freq = 1.0 / (10000 ** (torch.arange(0, d, 2).float().to(device) / d))
    t = torch.arange(T, device=device).type_as(inv_freq)
    freqs = torch.einsum('i,j->ij', t, inv_freq) # (T, d/2)

    # Create sin/cos embeddings
    emb = torch.cat((freqs, freqs), dim=-1) # (T, d)
    cos = emb.cos() # (T, d)
    sin = emb.sin() # (T, d)

    # Rotate: [-x1, x0, -x3, x2...]
    x_rotated = torch.cat((-x[..., 1::2], x[..., 0::2]), dim=-1)

    # RoPE formula: x * cos + x_rotated * sin
    return (x * cos) + (x_rotated * sin)


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # apply rope
        k = RoPE(k, self.head_size, x.device)
        q = RoPE(q, self.head_size, x.device)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head, block_size, dropout):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

#  simple  model
class LanguageModel(nn.Module):

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        # pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb #+ pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [33]:
import pandas as pd
# --- Fixed Hyperparameters ---
batch_size = 16
block_size = 32
max_iters = 5000
learning_rate = 1e-3
dropout = 0.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Part (c) Grid Search ---
n_layer_options = [4, 6, 8]
n_embd_options = [64, 128, 256]
n_head = 4 # Keep fixed to satisfy n_embd % n_head == 0

results_list = []

print("Starting Hyperparameter Search...")
print(f"{'n_layer':<10} | {'n_embd':<10} | {'Final Loss':<15}")
print("-" * 40)

for nl in n_layer_options:
    for ne in n_embd_options:

        # 1. Initialize fresh model and optimizer
        model = LanguageModel(
            vocab_size=vocab_size, # Ensure vocab_size is defined from your dataset
            n_embd=ne,
            n_head=n_head,
            n_layer=nl,
            block_size=block_size,
            dropout=dropout
        ).to(device)

        # print the number of parameters in the model
        print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

        import time

        start_time = time.time()

        # 2. Training loop for this configuration

        for iter in range(max_iters):

            # every once in a while evaluate the loss on train and val sets
            if iter % eval_interval == 0 or iter == max_iters - 1:
                losses = estimate_loss()
                print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

            # sample a batch of data
            xb, yb = get_batch('train')

            # evaluate the loss
            logits, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"Total time taken: {elapsed_time} seconds")


        # 3. Record the final loss
        final_loss = loss.item()
        results_list.append({
            'n_layer': nl,
            'n_embd': ne,
            'final_pretraining_loss': round(final_loss, 4)
        })

        print(f"{nl:<10} | {ne:<10} | {final_loss:.4f}")

        # generate text
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
        print(context)
        print("Text generation test")
        print("-" * 20)
        print(encoding.decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
        # 4. Cleanup memory to avoid Colab Out-of-Memory (OOM) errors
        del model
        torch.cuda.empty_cache()

# --- Final Table Generation ---
df = pd.DataFrame(results_list)
print("\n---Final Results ---")
print(df.to_string(index=False))

Starting Hyperparameter Search...
n_layer    | n_embd     | Final Loss     
----------------------------------------
13.135029 M parameters
step 0: train loss 11.6845, val loss 11.6870
step 100: train loss 5.9529, val loss 7.3938
step 200: train loss 4.6252, val loss 7.0797
step 300: train loss 3.3779, val loss 7.0396
step 400: train loss 2.4682, val loss 7.2348
step 500: train loss 1.7350, val loss 7.5409
step 600: train loss 1.2046, val loss 7.9475
step 700: train loss 0.8197, val loss 8.3104
step 800: train loss 0.5909, val loss 8.6390
step 900: train loss 0.4653, val loss 8.8669
step 1000: train loss 0.3574, val loss 9.0387
step 1100: train loss 0.3075, val loss 9.2437
step 1200: train loss 0.2834, val loss 9.4083
step 1300: train loss 0.2595, val loss 9.5164
step 1400: train loss 0.2432, val loss 9.5891
step 1500: train loss 0.2268, val loss 9.7045
step 1600: train loss 0.2168, val loss 9.6979
step 1700: train loss 0.1982, val loss 9.8946
step 1800: train loss 0.1987, val loss 9.9